# Jamo — 굵은 자모 마스크 조건부 분리 학습
GPU 런타임에서 위부터 실행하세요. 기존 ReferenceStrokeModel과 학습 코드를 재사용하며 기준 마스크를 skeleton으로 변환하지 않습니다.

한 채널은 하나의 **자모/분리 단위**입니다. 기준과 정답 목록은 같은 순서여야 합니다. 기존 skeleton CSV를 지정하면 기존 데이터로 학습되므로, 굵은 자모 마스크를 가리키는 CSV를 지정하세요. 단순히 skeleton을 굵게 만드는 노트북은 아닙니다.

[데이터 준비 안내](https://github.com/yechan25/hsqm/blob/main/docs/stroke_model/JAMO.md) · [CSV 예시](https://github.com/yechan25/hsqm/blob/main/configs/jamo_paths_example.csv)


In [ ]:
from pathlib import Path
import sys, subprocess, tempfile, json
root = Path(tempfile.mkdtemp(prefix="jamo_", dir="/content"))
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/yechan25/hsqm.git", str(root)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(root / "requirements.txt")], check=True)
sys.path.insert(0, str(root))
for key in list(sys.modules):
    if key == "stroke_model" or key.startswith("stroke_model."):
        del sys.modules[key]
commit = subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True).strip()
print("Jamo 코드 준비 완료 / GitHub 버전:", commit)


## 데이터와 학습 설정
아래 TRAIN_CSV를 실제 자모 데이터 CSV 경로로 바꾸세요. 기본 경로는 새 데이터용 예시이며 자동 생성되지 않습니다. 같은 경로의 이미지를 수정했다면 REBUILD=True로 설정하세요.

각 마스크는 잘라낸 자모 그림이 아니라 **전체 기준/입력 글자와 같은 캔버스에서 해당 자모만 남긴 이미지**여야 합니다. 한 글자의 기준과 정답에 자모가 2개면 두 목록 모두 2개입니다.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
TRAIN_CSV = "/content/drive/Shareddrives/2026 자율연구/HSQM/dataset/jamo/train_dataset/paths.csv"
TEST_CSV = ""  # 선택: 학습/검증과 분리된 자모 test CSV
OUTPUT_ROOT = "/content/drive/Shareddrives/2026 자율연구/HSQM/jamo"
REBUILD = False
VARIANTS = 20
EPOCHS = 30
BATCH_SIZE = 8  # GPU 메모리가 부족하면 4로 변경
SEED = 42


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from stroke_model.convert_reference_csv import existing_path
from stroke_model.reference_cache import prepare_cached
from stroke_model.preview_reference import preview_reference
from stroke_model.train_reference import read_manifest, StrokeDataset

try:
    csv_path = existing_path(TRAIN_CSV)
except FileNotFoundError as exc:
    raise FileNotFoundError(
        "TRAIN_CSV를 굵은 자모 기준/정답 마스크가 준비된 실제 CSV 경로로 바꾸세요. "
        "형식: 저장소 configs/jamo_paths_example.csv"
    ) from exc
output_root = Path(OUTPUT_ROOT)
manifest = prepare_cached(csv_path, output_root / "data", seed=SEED, rebuild=REBUILD)
report = json.loads((manifest.parent / "conversion_report.json").read_text(encoding="utf-8"))
annotations = report["rows"]
foreground = sum(r["foreground_pixels"] for r in annotations)
excluded = sum(r["overlap_pixels"] + r["missing_pixels"] for r in annotations)
print(f"정답 전경 커버리지: {1 - excluded / max(foreground, 1):.1%}")
print("겹침/미할당 픽셀은 학습과 Dice 평가에서 제외됩니다.")
rows = read_manifest(manifest)
dataset = StrokeDataset(rows, size=128)
unit_counts = []
for row in rows:
    with np.load(row["path"]) as sample:
        unit_counts.append(sample["strokes"].shape[0])
if max(unit_counts) < 2:
    raise ValueError("모든 샘플의 자모가 1개입니다. 현재 softmax 분리 모델은 K=1이면 입력 전체를 그대로 반환합니다. 여러 자모로 나눌 글자와 대응 마스크를 준비하세요.")
print("샘플별 자모 수:", sorted(set(unit_counts)))
# 모델에 실제로 들어가는 기준 마스크와 대응 정답을 먼저 확인합니다.
for index in range(min(3, len(dataset))):
    image, reference, strokes, labels = dataset[index]
    for start in range(0, len(strokes), 6):
        ids = list(range(start, min(start + 6, len(strokes))))
        fig, axes = plt.subplots(2, len(ids) + 1, squeeze=False,
                                 figsize=(2.5 * (len(ids) + 1), 5))
        axes[0, 0].imshow(reference[0], cmap="gray_r", vmin=0, vmax=1)
        axes[0, 0].set_title("Reference")
        axes[1, 0].imshow(image[0], cmap="gray_r", vmin=0, vmax=1)
        axes[1, 0].set_title("Input")
        for col, k in enumerate(ids, 1):
            axes[0, col].imshow(strokes[k], cmap="gray_r", vmin=0, vmax=1)
            axes[0, col].set_title(f"Reference jamo {k+1}")
            axes[1, col].imshow(labels == k, cmap="gray_r", vmin=0, vmax=1)
            axes[1, col].set_title(f"Target jamo {k+1}")
        for ax in axes.flat:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close(fig)
preview_reference(manifest, count=3)


In [ ]:
from stroke_model.reference_cache import ensure_augmented
training_manifest = ensure_augmented(manifest, variants=VARIANTS, seed=SEED)
print("학습 manifest:", training_manifest)


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Colab 메뉴: 런타임 → 런타임 유형 변경 → GPU를 선택하세요.")
output = output_root / ("jamo_run_" + datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M%S_%f"))
output.mkdir(parents=True, exist_ok=False)
settings = dict(experiment="jamo", code_commit=commit, train_csv=str(csv_path),
                manifest=str(manifest), training_manifest=str(training_manifest),
                variants=VARIANTS, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED)
(output / "run_config.json").write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding="utf-8")
cmd = [sys.executable, "-u", "-m", "stroke_model.train_reference",
       "--manifest", str(training_manifest), "--epochs", str(EPOCHS),
       "--batch-size", str(BATCH_SIZE), "--num-workers", "2", "--seed", str(SEED),
       "--local-cache", "/content/jamo_local", "--output", str(output)]
print("Drive 데이터를 로컬로 복사한 뒤 학습합니다. 체크포인트 저장:", output, flush=True)
with subprocess.Popen(cmd, cwd=root, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        result = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
        raise
if result:
    raise RuntimeError(f"학습 실패: 종료 코드 {result}. 위 로그를 확인하세요.")
print("체크포인트:", output / "best.pt")


## 학습 결과
검증 Dice는 정답이 지정된 픽셀에 대한 점수입니다. 회색으로 표시되는 미할당 영역은 평가에서 제외됩니다. 자모로 묶으면 과제가 달라지므로 기존 개별 획 Dice와 직접 비교하지 마세요.


In [ ]:
import pandas as pd
history = pd.read_json(output / "history.json")
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history["epoch"], history["train_loss"])
axes[0].set(title="Training loss", xlabel="Epoch")
axes[1].plot(history["epoch"], history["val_macro_dice"])
axes[1].set(title="Validation Dice (annotated only)", xlabel="Epoch", ylim=(0, 1))
plt.tight_layout()
fig.savefig(output / "learning_curves.png", dpi=150)
plt.show()
plt.close(fig)
display(history.loc[[history["val_macro_dice"].idxmax()]])
preview_reference(manifest, checkpoint=output / "best.pt", count=5,
                  save_dir=output / "val_preview", show_soft=True)
print("학습 곡선, 비교 이미지, soft mask 저장:", output)


## 선택: 별도 test 데이터 평가
TEST_CSV가 비어 있으면 건너뜁니다. 학습/검증 CSV를 test로 다시 지정하지 마세요. 동일 경로의 test 이미지를 수정했다면 새 CSV 파일명으로 저장하여 기존 test 캐시와 구분하세요.


In [ ]:
if TEST_CSV.strip():
    from stroke_model.test_reference import run_test
    test_csv = existing_path(TEST_CSV)
    if test_csv.resolve() == csv_path.resolve():
        raise ValueError("학습 CSV와 다른 held-out test CSV를 지정하세요.")
    test_metrics = run_test(output / "best.pt", test_csv, preview_count=5)
else:
    print("별도 test CSV 미지정: 위 결과는 검증 세트 결과입니다.")
